# Person 1 — DBSCAN Clustering & Epsilon Tuning

**Goal:** Cluster NYC taxi pickup locations using DBSCAN to identify high-density ride request zones, then tune epsilon for pooling quality.

**Features used for clustering:**
- `pickup_lat`, `pickup_lon` — geographic position
- `pickup_hour` — time-of-day (engineered from `pickup_datetime`)

**Outputs handed off to Person 3:**
- `df_clustered` with a `dbscan_cluster` column (-1 = noise/unclustered)
- `dbscan_summary` dict: `n_clusters`, `noise_pct`, `silhouette_score`

**Workflow:**
1. Load 10k-row subset  
2. Feature engineering  
3. k-distance plot → choose epsilon  
4. Run DBSCAN, report metrics  
5. Tune epsilon / min_samples  
6. Scale up to full dataset once tuned

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

plt.rcParams["figure.dpi"] = 120

## 1. Load Data (10k-row subset)

In [ ]:
DATA_PATH = "../Data/FHVTrip_VehicleEmissions_Merged_2023.csv"
SUBSET_N  = 10_000  # start small; increase once tuning is complete

df_full = pd.read_csv(DATA_PATH, low_memory=False)

# Reproducible subset — use a fixed random seed
df = df_full.sample(n=SUBSET_N, random_state=42).reset_index(drop=True)

# Coerce types
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce")
df["pickup_lat"]      = pd.to_numeric(df["pickup_lat"],      errors="coerce")
df["pickup_lon"]      = pd.to_numeric(df["pickup_lon"],      errors="coerce")

# Drop rows missing key fields
df = df.dropna(subset=["pickup_lat", "pickup_lon", "pickup_datetime"])
print(f"Working subset: {len(df):,} rows")

## 2. Feature Engineering

In [ ]:
df["pickup_hour"] = df["pickup_datetime"].dt.hour

# Feature matrix: geographic + temporal
# Cyclic encoding of hour so midnight wraps correctly
df["hour_sin"] = np.sin(2 * np.pi * df["pickup_hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["pickup_hour"] / 24)

features = ["pickup_lat", "pickup_lon", "hour_sin", "hour_cos"]

X_raw = df[features].values

# Standardize so lat/lon and hour are on comparable scales
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"Feature matrix shape: {X.shape}")
pd.DataFrame(X, columns=features).describe().round(3)

## 3. k-Distance Plot — Choose Epsilon

Rule of thumb: `min_samples = 2 * n_features`.  
The "elbow" in the sorted k-distance curve gives a good starting epsilon.

In [ ]:
MIN_SAMPLES = 2 * X.shape[1]   # = 8 for 4 features

nbrs = NearestNeighbors(n_neighbors=MIN_SAMPLES, algorithm="ball_tree").fit(X)
distances, _ = nbrs.kneighbors(X)
k_distances   = np.sort(distances[:, -1])[::-1]   # distance to k-th neighbor, descending

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_distances, linewidth=1.2)
ax.set_xlabel("Points sorted by distance")
ax.set_ylabel(f"{MIN_SAMPLES}-NN distance (standardized)")
ax.set_title("k-Distance Plot — the 'knee' suggests a good epsilon")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("kdistance_plot.png", bbox_inches="tight")
plt.show()

# TODO: visually pick the knee value and set EPSILON below

## 4. Run DBSCAN

In [ ]:
# --- TUNE THESE ---
EPSILON     = 0.5   # ← adjust based on k-distance knee above
MIN_SAMPLES = 8     # ← 2 * n_features is a good default

db = DBSCAN(eps=EPSILON, min_samples=MIN_SAMPLES)
df["dbscan_cluster"] = db.fit_predict(X)

n_clusters  = len(set(df["dbscan_cluster"])) - (1 if -1 in df["dbscan_cluster"].values else 0)
noise_pct   = (df["dbscan_cluster"] == -1).mean() * 100
clustered   = df[df["dbscan_cluster"] != -1]

# Silhouette score only meaningful when >1 cluster and <all noise
if n_clusters > 1 and len(clustered) > 1:
    sil = silhouette_score(X[clustered.index], clustered["dbscan_cluster"], sample_size=5000, random_state=42)
else:
    sil = float("nan")

print(f"eps={EPSILON}, min_samples={MIN_SAMPLES}")
print(f"  Clusters found  : {n_clusters}")
print(f"  Noise points    : {noise_pct:.1f}%")
print(f"  Silhouette score: {sil:.4f}" if not np.isnan(sil) else "  Silhouette score: N/A")

## 5. Epsilon Grid Search

Sweep a range of epsilon values and record metrics — share results with Person 2 for comparison.

In [ ]:
eps_values   = [0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]
tune_results = []

for eps in eps_values:
    labels   = DBSCAN(eps=eps, min_samples=MIN_SAMPLES).fit_predict(X)
    n_cl     = len(set(labels)) - (1 if -1 in labels else 0)
    noise    = (labels == -1).mean() * 100
    mask     = labels != -1
    sil_val  = silhouette_score(X[mask], labels[mask], sample_size=5000, random_state=42) if n_cl > 1 and mask.sum() > 1 else float("nan")
    tune_results.append({"eps": eps, "n_clusters": n_cl, "noise_pct": round(noise, 1), "silhouette": round(sil_val, 4) if not np.isnan(sil_val) else None})

tune_df = pd.DataFrame(tune_results)
print(tune_df.to_string(index=False))

## 6. Visualize Clusters on Map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

noise_mask   = df["dbscan_cluster"] == -1
cluster_mask = ~noise_mask

ax.scatter(
    df.loc[noise_mask, "pickup_lon"],
    df.loc[noise_mask, "pickup_lat"],
    c="lightgrey", s=4, alpha=0.4, label="Noise"
)
scatter = ax.scatter(
    df.loc[cluster_mask, "pickup_lon"],
    df.loc[cluster_mask, "pickup_lat"],
    c=df.loc[cluster_mask, "dbscan_cluster"],
    cmap="tab20", s=6, alpha=0.7
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"DBSCAN Clusters (eps={EPSILON}, min_samples={MIN_SAMPLES}) — {n_clusters} clusters")
ax.legend(loc="upper right")
plt.colorbar(scatter, ax=ax, label="Cluster ID")
plt.tight_layout()
plt.savefig("dbscan_map.png", bbox_inches="tight")
plt.show()

## 7. Export Results for Person 3

In [ ]:
# Save labelled subset for poolability analysis
df.to_csv("../Data/trips_dbscan_labeled.csv", index=False)
print(f"Saved {len(df):,} rows → ../Data/trips_dbscan_labeled.csv")

dbscan_summary = {
    "algorithm":   "DBSCAN",
    "epsilon":      EPSILON,
    "min_samples":  MIN_SAMPLES,
    "n_clusters":   n_clusters,
    "noise_pct":    round(noise_pct, 2),
    "silhouette":   round(sil, 4) if not np.isnan(sil) else None,
    "subset_n":     len(df),
}
print(dbscan_summary)

## 8. Scale Up (run after tuning is complete)

Uncomment and rerun once epsilon is confirmed on the 10k subset.

In [ ]:
# SCALE_N = 100_000   # next step
# # SCALE_N = len(df_full)   # full dataset

# df_scaled = df_full.sample(n=SCALE_N, random_state=42).reset_index(drop=True)
# df_scaled["pickup_datetime"] = pd.to_datetime(df_scaled["pickup_datetime"], errors="coerce")
# df_scaled["pickup_hour"]     = df_scaled["pickup_datetime"].dt.hour
# df_scaled["hour_sin"]        = np.sin(2 * np.pi * df_scaled["pickup_hour"] / 24)
# df_scaled["hour_cos"]        = np.cos(2 * np.pi * df_scaled["pickup_hour"] / 24)
# df_scaled = df_scaled.dropna(subset=["pickup_lat", "pickup_lon"])

# X_scaled = scaler.transform(df_scaled[["pickup_lat", "pickup_lon", "hour_sin", "hour_cos"]].values)
# df_scaled["dbscan_cluster"] = DBSCAN(eps=EPSILON, min_samples=MIN_SAMPLES).fit_predict(X_scaled)
# df_scaled.to_csv("../Data/trips_dbscan_labeled_full.csv", index=False)
# print(f"Saved {len(df_scaled):,} rows (scaled)")